In [2]:
# --- Import libraries ---
import os
import requests
from openai import OpenAI

In [3]:
# --- DeepSeek API key securely ---
os.environ["DEEPSEEK_API_KEY"] = "sk-c3973e8bf7954f0899ebd0330da648fd"

In [4]:
# client = OpenAI(api_key="sk-c3973e8bf7954f0899ebd0330da648fd", base_url="https://api.deepseek.com")
# print(client.models.list())

In [5]:
# --- DeepSeek API endpoint and model ---
DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"
# DEEPSEEK_MODEL = "deepseek-chat"
DEEPSEEK_MODEL = "deepseek-chat"

In [6]:
# --- Prepare Input Data ---
input_1 = """
Offshore wind turbines must adhere to Load Resistance Factor Design (LRFD) principles.
IEC standards currently specify a partial safety factor of 1.35,
but in hurricane-prone areas of the U.S., API standards require additional robustness checks
using a 500-year return period for L2 structures. The discrepancy between IEC and API safety factors
for offshore wind turbines is an ongoing regulatory challenge.
"""

input_2 = """Regulation and compliance in U.S. waters can be under state or federal jurisdiction,
depending on the water body and the distance from shore. State jurisdiction applies to all the Great Lakes’ waters,
and, for most states, three nautical miles seaward (3.5 statute miles or 5.6 kilometers). The exceptions are Louisiana,
Texas, and the Gulf Coast of Florida. Specifically:
Louisiana extends 3 pre-1954 U.S nautical miles (3.455 miles or 5.560 kilometers) seaward.
Texas and the Florida Gulf Coast extend 9 U.S. nautical miles (10.4 miles or 16.7 kilometers) seaward.
"""

input_3 = """
API provides 1-hour, 10-minute, 1-minute, and 3-second wind averages for the Gulf of Mexico.
The 100-year extreme wind and wave conditions govern U.S. oil and gas development. In 2007, BOEM (formerly MMS)
updated its met-ocean criteria as a result of Hurricanes Ivan, Katrina, and Rita (spanning from 2004 to 2005)
when some offshore platforms suffered significant damage. The central section had the highest extreme values,
setting the 100-year 10-minute average mean wind speed at 10 m above water to 54.5 m/s.
"""

In [7]:
from prompt_deepseek import prompt as base_prompt

In [8]:
# --- Format the prompt ---
formatted_prompt = base_prompt.format(DOCUMENTATION=input_2)

In [9]:
# --- DeepSeek query function ---
def deepseek_query(prompt, max_tokens=500):
    headers = {
        "Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}",
        "Content-Type": "application/json"
    }
    data = {
        "model": DEEPSEEK_MODEL,
        "messages": [
            {"role": "user", "content": str(prompt)}
        ],
        "max_tokens": max_tokens
    }
    response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data)
    print("Status code:", response.status_code)
    print("Response text:", response.text)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

In [10]:
# --- Call DeepSeek and print only the output ---
try:
    deepseek_output = deepseek_query(formatted_prompt)
    print(deepseek_output.strip())
except Exception as e:
    print(f"Error during DeepSeek inference: {e}")

KeyboardInterrupt: 

### Work on CSV

In [ ]:
import pandas as pd

# --- Load in CSV file ---
csv_path = "/Users/li/Downloads/27.csv"  # Change to your actual CSV file path
df = pd.read_csv(csv_path)

# --- Prepare a list to store results ---
results = []

# --- Process each row ---
for idx, row in df.iterrows():
    content = row['content']
    formatted_prompt = base_prompt.format(DOCUMENTATION=content)
    try:
        deepseek_output = deepseek_query(formatted_prompt, max_tokens=2000)  # Increase max_tokens if needed
        results.append({
            "document_id": row['document_id'],
            "page_number": row['page_number'],
            "output": deepseek_output.strip()
        })
    except Exception as e:
        results.append({
            "document_id": row['document_id'],
            "page_number": row['page_number'],
            "output": f"Error: {e}"
        })

# --- Convert results to DataFrame and save ---
results_df = pd.DataFrame(results)
results_df.to_csv("deepseek_outputs.csv", index=False)
print("Processing complete. Results saved to deepseek_outputs.csv.")

# Process JSON

In [ ]:
import json
from prompt_deepseek import prompt as base_prompt

with open("project_folder/LLM/Preprocessing/27.json") as f:
    chunks = json.load(f)

for chunk in chunks:
    formatted_prompt = base_prompt.format(DOCUMENTATION=chunk["text"])
    output = deepseek_query(formatted_prompt)
    print(output)

Status code: 200
Response text: {"id":"ef82f2da-31a2-46fb-8e7f-28ab8ae71371","object":"chat.completion","created":1746460955,"model":"deepseek-chat","choices":[{"index":0,"message":{"role":"assistant","content":"```json\n{\n  \"document_metadata\": {\n    \"title\": \"Revolution Wind, LLC Outer Continental Shelf Preconstruction Air Permit No. OCS-R1-05\",\n    \"document_number\": \"OCS-R1-05\",\n    \"Type of wind farm\": \"offshore\"\n  },\n  \"regulatory_constraints\": [\n    {\n      \"type\": \"Jurisdictional\",\n      \"requirement\": \"Construction and operation of up to 100 wind turbine generators (WTGs) and up to 2 Offshore Substations (OSSs) within federal waters on the Outer Continental Shelf (OCS)\",\n      \"scope\": \"Bureau of Ocean Energy Management (BOEM) Renewable Energy Lease Area OCS-A 0486 in the Rhode Island-Massachusetts Wind Energy Area\",\n      \"numerical_value\": null,\n      \"unit\": null,\n      \"source\": \"Clean Air Act (CAA) Section 328 and 40 C.F.R. 

KeyboardInterrupt: 